# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All entities such as record sets, fields, and columns are referenced by their `@id`.

### Dataset Source
The dataset source is described by a Croissant schema and is accessible at:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
We load the metadata and dataset using `mlcroissant`. This allows us to explore the data schema before examining the actual content.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# URL to Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is a DatasetMetadata object
# Print the dataset high-level info
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
List available record sets and their fields. All entities are referenced by their `@id`.

In [ ]:
# Inspect the available record sets, each by its @id and fields
print("Record sets in the dataset (referenced by @id):")
record_sets = metadata.record_sets
for rset in record_sets:
    print(f"- @id: {rset.id} | name: {rset.name}")
    print("  Fields:")
    for field in rset.fields:
        print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
    print()

## 3. Data Extraction
We extract data from one or more record sets into Pandas DataFrames. All record sets and fields are referenced by their `@id`.

We demonstrate this for all available record sets.

In [ ]:
# Collect all record set @ids from the metadata
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

print(f"Found Record Sets: {record_set_ids}")

# Load all records from all record sets into pandas DataFrames
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for record set @id={rs_id}. Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    if df.shape[0] > 0:
        display(df.head(2))
    print()

## 4. Exploratory Data Analysis (EDA)
Let's pick one record set for in-depth examination. We'll select the main protein abundance/annotation table, which typically contains numeric columns. 

**Note:** Replace `<selected_record_set_id>` and field IDs below with the `@id` values you prefer if exploring a different table.

In [ ]:
# Pick the main record set (for this dataset, usually the largest non-metadata table)
# We'll heuristically pick the largest DataFrame (biggest number of columns and rows)
selected_record_set_id = None
maxrows = 0
for rs_id, df in dataframes.items():
    if df.shape[0] > maxrows:
        selected_record_set_id = rs_id
        maxrows = df.shape[0]

print(f"Selected main table for EDA: @id={selected_record_set_id}")
df = dataframes[selected_record_set_id]
display(df.head(3))

# Show column @ids and infer numeric fields (try to guess float/int columns)
numeric_fields = []
for field in [f for f in metadata.record_set(selected_record_set_id).fields]:
    if field.data_type in ('Float','Integer','Number') or ('float' in str(df[field.id].dtype) or 'int' in str(df[field.id].dtype)):
        numeric_fields.append(field.id)
print(f"Numeric fields by @id: {numeric_fields}")

# For demonstration, select the first numeric field
if len(numeric_fields) > 0:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field @id: {numeric_field_id}")
else:
    raise ValueError('No numeric field found for EDA — check your metadata!')

# Filter records based on a reasonable threshold for this field (e.g., > 10)
threshold = df[numeric_field_id].dropna().quantile(0.75)  # 75th percentile as example

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (top 25%): {len(filtered_df)} records")
display(filtered_df.head())

# Normalize the column
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized values for {numeric_field_id}:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try groupby by a non-numeric field if present (e.g., a categorical variable, try for a text or group field)
non_numeric_fields = [field.id for field in metadata.record_set(selected_record_set_id).fields if field.data_type in ('Text','String')]
group_field_id = non_numeric_fields[0] if non_numeric_fields else None
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Group mean for {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version. Relationships between fields can also be shown if enough suitable columns are present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, color='orange')
plt.title(f"Normalized {numeric_field_id}")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel("Count")
plt.show()

# Optional: scatter plot between first two numeric fields
if len(numeric_fields) > 1:
    field2 = numeric_fields[1]
    plt.figure(figsize=(6, 6))
    sns.scatterplot(data=filtered_df, x=numeric_field_id, y=field2)
    plt.title(f"Scatter: {numeric_field_id} vs {field2}")
    plt.show()

## 6. Conclusion
We demonstrated how to load the FAIR² dataset, inspect its schema via record set and field `@id`s, extract data, filter and normalize numeric fields, and visualize distributions. The `mlcroissant` library makes it easy to handle complex metadata-driven scientific datasets in a reproducible way.

**Further exploration:** You can perform deeper statistical analysis, join other record sets using keys (referenced by `@id`), or explore advanced visualization and machine learning workflows.